# Parametric Studies - Demo_BESS Generalized Sweep

This notebook sweeps one parameter of `Demo_BESS`, simulating the model at each value and comparing a chosen output variable across the sweep.

Set `REINITIALIZE_EACH_CASE` to choose whether the model is initialized once before the sweep, or reinitialized after each parameter update.

In [ ]:
import logging
import os
import shutil
import sys

sys.path.append(os.path.abspath(".."))

logging.getLogger("OMPython").setLevel(logging.ERROR)

import pandas as pd
from OMPython import ModelicaSystemOMC, OMCSessionLocal

from scripts.helpers import *
from scripts.parametric_study_helpers import case_label, load_modelica_file
from scripts.single_model_initialization import initialize_loaded_model

## User Configuration

Set the sweep component, parameter, values, and the model paths here.

In [ ]:
BASE_MODEL = "Demo_BESS"
BASE_MODEL_FILE = os.path.abspath(os.path.join("..", "FullWorkflow", "models", "Demo_BESS.mo"))

OMLIB_DIR = os.path.join(os.path.expanduser("~"), ".openmodelica", "libraries")
os.environ["OPENMODELICALIBRARY"] = OMLIB_DIR
MODELICA_PKG_PATH = os.path.join(OMLIB_DIR, "Modelica 3.2.3+maint.om", "package.mo")
DYNAWO_PKG_PATH = os.path.abspath(
    os.path.join("..", "..", "julia_openmodelica", "dynawo_library", "Dynawo", "package.mo")
)

SWEEP_COMPONENT = "BESS"
SWEEP_PARAMETER = "RMvHvPu"
SWEEP_VALUES = [0.0, 0.1, 0.3, 0.5]

REINITIALIZE_EACH_CASE = True

INIT_MODEL_BY_COMPONENT = {
    # "generatorSynchronous": "GeneratorSynchronousInt_INIT",
}

# Leave empty to disable slack-specific handling.
SLACK_COMPONENT = ""

PLOT_VARIABLE = "BESS.terminal.V.im"
OUTPUT_DIR = os.path.abspath(os.path.join("outputs", "generalized_parametric_runs"))

## Prepare OpenModelica Session

The output directories are created and the library paths are checked before the sweep runs.

In [ ]:
MODEL_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "modelica")
RESULT_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "results")

if not os.path.isfile(BASE_MODEL_FILE):
    raise RuntimeError(f"Base model file not found: {BASE_MODEL_FILE}")
if not os.path.isfile(DYNAWO_PKG_PATH):
    raise RuntimeError(f"Dynawo package not found: {DYNAWO_PKG_PATH}")
if not os.path.isfile(MODELICA_PKG_PATH):
    raise RuntimeError(f"Modelica package not found: {MODELICA_PKG_PATH}")

shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)
os.makedirs(RESULT_OUTPUT_DIR, exist_ok=True)

StudyOMC = OMCSessionLocal()
load_modelica_file(
    StudyOMC,
    BASE_MODEL_FILE,
    MODELICA_PKG_PATH,
    DYNAWO_PKG_PATH,
)
check_user_configuration_single(
    StudyOMC,
    model=BASE_MODEL,
    sweep_component=SWEEP_COMPONENT,
    sweep_parameter=SWEEP_PARAMETER,
    slack_component=SLACK_COMPONENT,
    init_model_by_component=INIT_MODEL_BY_COMPONENT,
)

## Initialize the Base Case

When `REINITIALIZE_EACH_CASE` is `False`, the base model is initialized once here and reused for every sweep value.

In [ ]:
if not REINITIALIZE_EACH_CASE:
    base_initialized_case = initialize_loaded_model(
        StudyOMC,
        source_model=BASE_MODEL,
        case_name=BASE_MODEL,
        output_dir=OUTPUT_DIR,
        modelica_package_path=MODELICA_PKG_PATH,
        dynawo_package_path=DYNAWO_PKG_PATH,
        init_model_by_component=INIT_MODEL_BY_COMPONENT,
        slack_component=SLACK_COMPONENT,
    )

    base_initialized_model = base_initialized_case["initialized_model"]
    base_initialized_model_file = base_initialized_case["initialized_model_file"]

StudyOMC.sendExpression("quit()")

## Run the Parameter Sweep

Each sweep value is applied, the model is simulated, and the selected variable is collected for comparison.

In [ ]:
sweep_results = []

for value in SWEEP_VALUES:
    label = case_label(value)
    parameter_name = SWEEP_PARAMETER.replace(".", "_")
    case_model = "_".join([BASE_MODEL, SWEEP_COMPONENT, parameter_name, label])
    selected_resultfile = os.path.join(RESULT_OUTPUT_DIR, case_model + "_selected.csv")

    StudyOMC = OMCSessionLocal()

    if REINITIALIZE_EACH_CASE:
        load_modelica_file(
            StudyOMC,
            BASE_MODEL_FILE,
            MODELICA_PKG_PATH,
            DYNAWO_PKG_PATH,
        )
        omc_call(
            StudyOMC,
            f"setParameterValue({BASE_MODEL}, {SWEEP_COMPONENT}.{SWEEP_PARAMETER}, {value})",
            parsed=False,
        )

        initialized_case = initialize_loaded_model(
            StudyOMC,
            source_model=BASE_MODEL,
            case_name=case_model,
            output_dir=OUTPUT_DIR,
            modelica_package_path=MODELICA_PKG_PATH,
            dynawo_package_path=DYNAWO_PKG_PATH,
            init_model_by_component=INIT_MODEL_BY_COMPONENT,
            slack_component=SLACK_COMPONENT,
        )

        simulation_model = initialized_case["initialized_model"]
        result_prefix = simulation_model
    else:
        load_modelica_file(
            StudyOMC,
            base_initialized_model_file,
            MODELICA_PKG_PATH,
            DYNAWO_PKG_PATH,
        )
        omc_call(
            StudyOMC,
            f"setParameterValue({base_initialized_model}, {SWEEP_COMPONENT}.{SWEEP_PARAMETER}, {value})",
            parsed=False,
        )

        simulation_model = base_initialized_model
        result_prefix = case_model

    StudySystem = ModelicaSystemOMC(session=StudyOMC)
    StudySystem.model(model_name=simulation_model)

    simflags = simulation_flags_without_log_stats(StudyOMC, simulation_model)
    omc_call(
        StudyOMC,
        f'simulate({simulation_model}, outputFormat="csv", '
        f'fileNamePrefix="{result_prefix}", simflags="{simflags}")',
        parsed=False,
    )

    raw_resultfile = os.path.join(str(StudySystem.getWorkDirectory()), result_prefix + "_res.csv")
    if PLOT_VARIABLE not in pd.read_csv(raw_resultfile, nrows=0).columns:
        raise RuntimeError(f'PLOT_VARIABLE "{PLOT_VARIABLE}" is not a variable in the simulation result.')
    df = pd.read_csv(raw_resultfile, usecols=["time", PLOT_VARIABLE])
    df["case"] = label
    df["parameter_value"] = value

    df.to_csv(selected_resultfile, index=False)
    sweep_results.append(df)
    StudyOMC.sendExpression("quit()")

    print(f"Finished case {label} with {SWEEP_COMPONENT}.{SWEEP_PARAMETER} = {value}")

sweep_df = pd.concat(sweep_results, ignore_index=True)

print("Stored selected result files in:", RESULT_OUTPUT_DIR)

## Compare Sweep Results

The selected variable is plotted across all sweep values.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots()
ax.set_title(f"Demo_BESS sweep: {SWEEP_COMPONENT}.{SWEEP_PARAMETER}")
ax.set_xlabel("Time (s)")
ax.set_ylabel(PLOT_VARIABLE)

for label, case_group in sweep_df.groupby("case"):
    value = case_group["parameter_value"].iloc[0]
    ax.plot(case_group["time"], case_group[PLOT_VARIABLE], label=f"{SWEEP_PARAMETER} = {value}")

ax.legend(loc="lower right")